In [2]:
import sys
import numpy as np
from collections import deque

'''
转换成二进制，为后续与原算法对比做准备
'''

def main():
   
    f = open("D(ABE3) - MANIERA (iJinjin) [Normal].osu", "r", encoding="utf8")
    content = deque(f.read().splitlines())
    f.close()

    filename, author_info = getName(content)
    timing = getTime(content)
    notes = getNotes(content)
    
    print(filename, "Conversion Success:", convert(notes, filename, author_info))
    # np.set_printoptions(threshold=np.inf)
    # print(notes)
   
    return

# 读取元数据，存为键值对
def getName(content):
    while content[0] != "[Metadata]":
        content.popleft()
    
    # 读作者信息
    filename = "_".join([content[10][13:], content[1][6:], content[3][7:], content[6][8:]])
    filename = filename.replace(" ", '')
    filename = filename.replace("/", "_")
    author_info = "The creator of this beatmap is " + content[5][8:] + ". Check out the original beatmap at osu.ppy.sh/s/" + content[10][13:] + "\n"

    while content[0] != "[Difficulty]":
        content.popleft()

    return filename, author_info    
    
# 读取时间轴，存为键值对
def getTime(content):
    while content[0] != "[TimingPoints]":
        content.popleft()
    content.popleft()
    
    output = []
    while content[0] != "":
        line = content.popleft().split(',')
        time = int(float(line[0]))
        value = float(line[1])
        # uninhereted timing point
        if value >= 0:
            output.append( (time, value, 1) )
        else:
            output.append( (time, None, value) )
    
    return output
    
    

# 读取击打物件，存为数组
def getNotes(content):
    while content[0] != "[HitObjects]":
        content.popleft()
    content.popleft()

    output = []

    while len(content):
        time = None
        hitObject = None
        endPoint = None
        line = content.popleft().replace(":", ",").split(",")
        try:
            time = int(float(line[2]))
            objectType = int(line[3])
        except Exception:
            print("new line encountered, continuing")
            continue

         # 读一下Holds，mania的谱子没有slider和spinner
        if objectType == 5:
            # 5是因为1+4了，是圆圈，把换色标记去掉
            hitObject = 1
        elif objectType == 1:
            # 5是因为1+4了，是圆圈，把换色标记去掉
            hitObject = 1
        elif objectType == 132:
            # 同理，这是128+4，是长键，把换色标记去掉
            hitObject = 128
            endPoint = int(float(line[5]))
        elif objectType == 128:
            # 同理，这是128+4，是长键，把换色标记去掉
            hitObject = 128
            endPoint = int(float(line[5]))
        else:
            # 不是的话就见鬼了
            print("HEY THIS CIRCLE DIDN'T GET READ!!!! WHY!!!!")

        output.append((time, hitObject, endPoint))
    return deque(output)

def convert(notes, filename, author_info):
    '''
    转换成二进制，输出一个bool表示转换是否成功
    在所有有音符的时间点标注一个one，没有的标注zero，以23ms为一个单位
    '''
    INCREMENT = 23
    
    time_output = []
    bin_output = []
    zero = 0
    one = 1
    np_outfile = filename + ".npy"
    npbin_outfile = filename +"_bin.npy"
    
    
    i = 0
    success = True
    try:
        while len(notes):
            curr_note = notes.popleft()
            time_output.append(curr_note[0])
            # 若窗口内已经读完，将窗口向后移一步
            while(curr_note[0] - i) > INCREMENT:
                bin_output.append(zero)
                i += INCREMENT
                
            hitObject = curr_note[1]
            if hitObject == 1:
                bin_output.append(one)
            else:
                # 否则是滑条
                endPoint = curr_note[2]
                bin_output.append(one)
                i += INCREMENT
                while endPoint - i > INCREMENT:
                    bin_output.append(zero)
                    i += INCREMENT
                bin_output.append(zero)
                i += INCREMENT
    except Exception as e:
        print(e.args[0])
        success = False
        
    bin_out = np.array(bin_output, dtype=np.int32)
    print(len(bin_out), "length of bin output array")
    np.save(npbin_outfile, bin_out)
    time_out = np.array(time_output, dtype=np.int32)
    print(len(time_out), "length of time output array")
    np.save(np_outfile, time_out)
        
    return success


if __name__ == "__main__":
    main()

5799 length of bin output array
654 length of time output array
272871_MANIERA_D(ABE3)_Normal Conversion Success: True
